# Enigma connector for Kaggle (dual T4) — Stokastisk 0.1.0 build 5

**You run this notebook yourself.** Nothing here starts automatically, and nothing opens an inbound port: the connector only makes *outbound HTTPS* calls to your Enigma API and talks to vLLM on `127.0.0.1`.

Use only within Kaggle's terms for **authorised personal experimentation**. Kaggle is not an always-on inference service; the session will end and Enigma will show the runtime as offline.

Settings: Accelerator **GPU T4 x2**, Internet **ON**, Python 3.13. Add a Kaggle **Secret** named `ENIGMA_RUNTIME_TOKEN` (create the runtime in Enigma with *Mock* turned **off**, then copy its key once). Attach the folder containing `enigma_connector/` as a Kaggle Dataset named `enigma-connector`.

Status: build 5 has **not** been run on a Kaggle GPU yet. Complete the checklist at the end and record the result in `docs/TEST_REPORT.md`.

In [ ]:
# REMOTE-ONLY GUARD: this notebook must never run on a developer laptop. It needs Kaggle's own runtime and exactly 2 x Tesla T4 (SM75).
import os
assert os.path.isdir("/kaggle/working"), "Not a Kaggle session: refusing to touch any local GPU. Run this notebook only on Kaggle."
assert os.environ.get("ENIGMA_FORBID_LOCAL_GPU") != "1", "ENIGMA_FORBID_LOCAL_GPU is set: this is a no-GPU host."
os.environ["ENIGMA_REMOTE_GPU_HOST"] = "1"


## 1. Install Stokastisk (skip if already installed)
Pinned by URL and SHA256, PyPI as the only index, so Kaggle's torch/triton stay untouched (command copied from the SDK's INSTALL.md).

In [ ]:
!python -m pip install --index-url https://pypi.org/simple "stokastisk @ https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl#sha256=fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e"

## 2. Verify the environment (read-only)

In [ ]:
import sys, importlib, importlib.metadata as md
assert sys.version_info[:2] == (3, 13), f"build 5 is a cp313 wheel; this kernel runs Python {sys.version.split()[0]}. Set the Kaggle notebook to Python 3.13 (or a 3.13 environment) before continuing."
try:
    import pynvml
except ImportError:
    get_ipython().system("python -m pip install --no-deps nvidia-ml-py")   # --no-deps: never touch torch/triton
    import pynvml
pynvml.nvmlInit()
n = pynvml.nvmlDeviceGetCount()
names = [pynvml.nvmlDeviceGetName(pynvml.nvmlDeviceGetHandleByIndex(i)) for i in range(n)]
caps = [pynvml.nvmlDeviceGetCudaComputeCapability(pynvml.nvmlDeviceGetHandleByIndex(i)) for i in range(n)]
pynvml.nvmlShutdown()
print("python", sys.version.split()[0], "| gpus:", n, names)
import stokastisk
print("stokastisk", stokastisk.__version__, "| prohibited_report clean:", stokastisk.prohibited_report()["clean"])
print("torch", md.version("torch"), "| triton", md.version("triton"), "| (these must be Kaggle's own pinned versions)")
print("compute capability:", caps)
assert n == 2 and all("T4" in (x.decode() if isinstance(x, bytes) else x) for x in names) and all(tuple(c) == (7, 5) for c in caps), "expected exactly 2 x Tesla T4 (SM75)"

## 3. Make the connector importable (no pip changes to torch/triton)

In [ ]:
import sys, importlib
sys.path.insert(0, "/kaggle/input/enigma-connector")
for mod, dist in (("prometheus_client", "prometheus_client"), ("pynvml", "nvidia-ml-py")):
    try:
        importlib.import_module(mod)
    except ImportError:
        # --no-deps: add only this package, never resolve/upgrade anything else
        get_ipython().system(f"python -m pip install --no-deps {dist}")
import enigma_connector
print("enigma_connector", enigma_connector.__version__)

## 4. Credentials (from Kaggle Secrets; the key is never printed)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["ENIGMA_RUNTIME_TOKEN"] = UserSecretsClient().get_secret("ENIGMA_RUNTIME_TOKEN")
os.environ["ENIGMA_API_URL"] = "https://YOUR-ENIGMA-API-HOST"   # <- edit: your deployed Enigma API base URL (https)
os.environ["ENIGMA_MODE"] = "stokastisk"
os.environ["ENIGMA_VLLM_URL"] = "http://127.0.0.1:8000"
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # the only model/mode with Kaggle evidence on the HTTP server path (TP=1, 2048 context)
# Optional API-cost tuning (default = unchanged 1 s claim poll). 5 s idle ceiling cuts claim polls by ~72 % (saves under USD 0.4/month in the cost model) but a prompt may wait up to ~5 s to be picked up.
# os.environ["ENIGMA_CLAIM_POLL_IDLE_S"] = "5"


## 5. Start vLLM through the SDK (explicit; TP=1, FP16, 2048 context)

In [ ]:
from stokastisk import StokastiskConfig
from stokastisk.serving import ServerProcess
server = ServerProcess(MODEL, StokastiskConfig(tensor_parallel_size=1, max_model_len=2048),
                       host="127.0.0.1", port=8000, served_model_name=MODEL)
server.start_and_wait(timeout=900)
print("healthy:", server.health(), "| log:", server.log_path)

## 6. Local smoke test (no Enigma involved)

In [ ]:
from stokastisk import serving
print("".join(serving.stream_chat(server.base_url, MODEL, [{"role": "user", "content": "Say hello in five words."}], max_tokens=24)))
m = serving.fetch_metrics(server.base_url)
print("vllm metric samples:", sum(k.startswith("vllm:") for k in m))

## 6b. Measured acceptance (still no Enigma involved)
Asserts real, measured behaviour: streamed chunks, native `/metrics` counters that actually move (parsed with `prometheus_client`), and per-GPU `nvidia-ml-py` readings for GPU 0 **and** GPU 1. TP=1 loads the model on GPU 0 only, so GPU 1 is expected to be idle but must still be reported.

In [ ]:
import json, time
from enigma_connector.gpu import NvmlSampler
from enigma_connector.vllm_metrics import VllmSampler          # parses vLLM's native /metrics with prometheus_client.parser

assert server.health(), "/health is not OK"
vs = VllmSampler(server.base_url)
before = vs.sample() or {}
chunks = list(serving.stream_chat(server.base_url, MODEL, [{"role": "user", "content": "Count from one to ten in words."}], max_tokens=64))
text = "".join(chunks)
time.sleep(2.5)                                                  # let vLLM publish the request to /metrics
after = vs.sample() or {}
assert len(chunks) > 1 and text.strip(), f"expected a streamed model reply, got {len(chunks)} chunk(s)"
gen_delta = (after.get("generation_tokens_total") or 0) - (before.get("generation_tokens_total") or 0)
assert gen_delta > 0, f"vllm:generation_tokens did not increase ({before.get('generation_tokens_total')} -> {after.get('generation_tokens_total')})"

nv = NvmlSampler()
inv, rows = nv.inventory(), nv.sample()
nv.close()
assert len(inv) == 2 and [r["gpu_index"] for r in rows] == [0, 1], "expected per-GPU readings for GPU 0 and GPU 1"
for r in rows:
    assert r["mem_total_mib"] and r["mem_total_mib"] > 14000, f"GPU {r['gpu_index']}: not a 15 GiB T4?"
assert rows[0]["mem_used_mib"] and rows[0]["mem_used_mib"] > 500, "GPU 0 shows no model memory: TP=1 model should be on GPU 0"
print("stream chunks:", len(chunks), "| generated-token counter +", gen_delta)
for g, r in zip(inv, rows):
    print(f"GPU{r['gpu_index']} {g['name']} mem {r['mem_used_mib']}/{r['mem_total_mib']} MiB util {r['util_gpu_pct']}% temp {r['temp_c']} C power {r['power_w']} W")
acceptance = {"stream_chunks": len(chunks), "generation_tokens_delta": gen_delta, "gpus": [{"index": r["gpu_index"], "name": g["name"], "mem_total_mib": r["mem_total_mib"],
              "mem_used_mib": r["mem_used_mib"], "util_gpu_pct": r["util_gpu_pct"]} for g, r in zip(inv, rows)]}

## 7. Connect to Enigma (outbound only)

In [ ]:
from enigma_connector.config import Config
from enigma_connector.cli import build
cfg = Config.from_env(mode="stokastisk", served_model=MODEL, tp=1, pp=1, max_model_len=2048)
connector = build(cfg)
connector.start()          # registers, then heartbeat + telemetry + job threads run in the background
print("connected as runtime", connector.runtime_id, "| run", connector.run_id)

Open Enigma → Playground and send a prompt. Re-run the next cell to watch counters.

In [ ]:
print(connector.stats, "| fatal:", connector.fatal, "| spooled batches:", len(connector.spool))

## 7b. Disconnect / offline / reconnect (optional, manual)
Run these in order and watch Enigma: (1) the runtime flips to **stale** then **Offline** after ~30-120 s without heartbeats, the chat is disabled, and **history stays visible**; (2) a prompt sent while offline is **not** answered by anything else (there is no fallback); (3) after reconnect the runtime is **Online** again and spooled telemetry is replayed in order.

In [ ]:
connector.stop()                                   # simulate the GPU host going away
print("connector stopped; check Enigma shows Offline and keeps history. Waiting 45 s ...")
import time; time.sleep(45)

In [ ]:
connector = build(Config.from_env(mode="stokastisk", served_model=MODEL, tp=1, pp=1, max_model_len=2048))
connector.start()                                  # a NEW run is registered; the old one is marked aborted
print("reconnected as runtime", connector.runtime_id, "| run", connector.run_id)

## 7c. Evidence record (secret-free) — paste into `docs/TEST_REPORT.md`
Prints only non-sensitive facts: versions, GPU model/memory, measured counters. It does **not** print the runtime token, API URL or any account detail.

In [ ]:
import json, platform, importlib.metadata as md2
evidence = {
    "host": "kaggle", "python": platform.python_version(),
    "stokastisk": stokastisk.__version__, "torch": md2.version("torch"), "triton": md2.version("triton"),
    "model": MODEL, "tp": 1, "max_model_len": 2048,
    "acceptance": acceptance,
    "connector_stats": {k: v for k, v in connector.stats.items() if isinstance(v, (int, float))},
    "connector_fatal": bool(connector.fatal), "spooled_batches": len(connector.spool),
}
print(json.dumps(evidence, indent=2))

## 8. Shut down cleanly

In [ ]:
connector.stop()
if connector.gpu is not None:
    connector.gpu.close()
print("server exit code:", server.stop(), "| shutdown report:", server.shutdown_report)

## GPU acceptance checklist (tick each, then record in docs/TEST_REPORT.md)

- [ ] Step 2: 2 x Tesla T4 visible; torch/triton versions are Kaggle's own (unchanged by step 1)
- [ ] `stokastisk.prohibited_report()["clean"]` is `True`; wheel SHA256 matches RELEASE.json
- [ ] Step 5: `/health` returns 200 for Qwen2.5-0.5B-Instruct TP=1, 2048 context
- [ ] Step 6: streamed text is model-generated and `/metrics` has `vllm:` samples
- [ ] Step 7: runtime shows **Online** and **not mock** in Enigma
- [ ] A Playground prompt streams, persists after reload, and shows measured TTFT / latency
- [ ] GPU 0 and GPU 1 charts show real NVML values; unsupported counters show N/A
- [ ] vLLM page shows counters moving after the request; no negative rates
- [ ] Stop button halts generation mid-stream
- [ ] Step 8: both GPUs release memory; Enigma shows the runtime **Offline** within ~2 minutes
- [ ] Revoking the key in Enigma stops the connector (`connector.fatal` is set)

TP=2 / PP=2 HTTP serving and other models are **not** covered by this checklist and remain unverified.
- [ ] Step 6b: streamed reply had > 1 chunk, `vllm:generation_tokens` increased, GPU 0 **and** GPU 1 reported by `nvidia-ml-py` (both ~15 GiB T4, GPU 0 holds the model for TP=1)
- [ ] Step 7b: runtime shows Offline when the connector stops, history persists, no fallback answers, runtime returns Online after reconnect
- [ ] Step 7c: evidence JSON pasted into `docs/TEST_REPORT.md` (no token, no URL)
- [ ] Shutdown is clean (`server.stop()` exit code 0) and **no AWS GPU or hosted model was used at any point**

Larger models and TP=2 are separate experiments and must be labelled **unverified** until run.


## 9. SEPARATE EXPERIMENT: TP=2 serving (UNVERIFIED, run only after sections 1-8 passed)
This is a **different experiment** from the TP=1 run above. A TP=1 result must never be reported as dual-GPU parallel inference, and a TP=2 result must be recorded under its own heading in `docs/TEST_REPORT.md`. The SDK's memory notes say measured TP=2 can be *slower* than TP=1 on this hardware; record the measurement, do not assume a speed-up.
Restart the kernel (or run section 8 first) so both GPUs are free before running the next cells.

In [ ]:
# TP=2 guard: both T4s must be (nearly) empty, otherwise the TP=1 server from above is still alive.
import pynvml
pynvml.nvmlInit()
used = [pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(i)).used // 2**20 for i in range(2)]
pynvml.nvmlShutdown()
assert max(used) < 500, f"GPU memory in use {used} MiB: stop the TP=1 server (section 8) or restart the kernel first"
from stokastisk import StokastiskConfig, serving
from stokastisk.serving import ServerProcess
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
server2 = ServerProcess(MODEL, StokastiskConfig(tensor_parallel_size=2, max_model_len=2048), host="127.0.0.1", port=8000, served_model_name=MODEL)
server2.start_and_wait(timeout=1200)
print("TP=2 healthy:", server2.health(), "| log:", server2.log_path)

In [ ]:
# TP=2 acceptance: BOTH GPUs must hold model memory. If GPU 1 stays idle this is NOT tensor-parallel serving.
import time
from enigma_connector.gpu import NvmlSampler
from enigma_connector.vllm_metrics import VllmSampler
vs2 = VllmSampler(server2.base_url); before = vs2.sample() or {}
chunks2 = list(serving.stream_chat(server2.base_url, MODEL, [{"role": "user", "content": "Count from one to ten in words."}], max_tokens=64))
time.sleep(2.5); after = vs2.sample() or {}
nv2 = NvmlSampler(); rows2 = nv2.sample(); nv2.close()
assert len(chunks2) > 1, "no streamed reply"
assert (after.get("generation_tokens_total") or 0) > (before.get("generation_tokens_total") or 0), "vllm generation counter did not move"
assert all((r["mem_used_mib"] or 0) > 500 for r in rows2), f"TP=2 requires model memory on BOTH GPUs: {[r['mem_used_mib'] for r in rows2]}"
tp2_evidence = {"experiment": "TP=2 (separate from TP=1)", "tp": 2, "model": MODEL, "stream_chunks": len(chunks2), "gpu_mem_used_mib": [r["mem_used_mib"] for r in rows2]}
print(tp2_evidence)

In [ ]:
# To run the Enigma connector against TP=2, build it with tp=2 so the dashboard labels the run correctly:
# connector2 = build(Config.from_env(mode="stokastisk", served_model=MODEL, tp=2, pp=1, max_model_len=2048)); connector2.start()
# ... send a Playground prompt, then:
# connector2.stop(); print("server exit code:", server2.stop())